In [1]:
import os

In [2]:
%pwd

'd:\\Data_Science\\Projects\\Chicken-Disease-Classification-\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'd:\\Data_Science\\Projects\\Chicken-Disease-Classification-'

In [5]:
import tensorflow as tf

In [6]:
model = tf.keras.models.load_model('artifacts/training/model.keras')

In [7]:
from dataclasses import dataclass
from pathlib import Path

In [10]:
@dataclass
class EvaluationConfig:
    path_of_model: Path
    training_data:Path
    all_params: dict
    params_image_size: list
    params_batch_size: int

In [12]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories,save_json

In [22]:
class ConfigurationManager:
    def __init__(self, config_file_path:str = CONFIG_FILE_PATH, params_file_path:str = PARAMS_FILE_PATH):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)

    def get_validation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model = self.config.training.trained_model_path,
            training_data = os.path.join(self.config.data_ingestion.unzip_dir,'Chicken-fecal-images'),
            all_params = self.params,
            params_image_size = self.params.IMAGE_SIZE,
            params_batch_size = self.params.BATCH_SIZE
        )
        return eval_config

In [26]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config


    def _valid_generator(self):
        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split = 0.30
        )

        dataflow_kwargs = dict(
            target_size = self.config.params_image_size[:-1],
            batch_size = self.config.params_batch_size,
            interpolation = 'bilinear'
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(**datagenerator_kwargs)
        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory = self.config.training_data,
            subset = 'validation',
            shuffle = False,
            **dataflow_kwargs
        )

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    
    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score =self.model.evaluate(self.valid_generator)

    def save_score(self):
        scores = {'loss' : self.score[0], 'accuracy': self.score[1]}
        save_json(path=Path('scores.json'), data=scores)


In [27]:
try:
    config = ConfigurationManager().get_validation_config()
    eval = Evaluation(config)
    eval.evaluation()
    eval.save_score()
except Exception as e:
    raise e

[2026-02-18 10:00:04,360 - INFO - common - yaml file: config\config.yaml loaded successfully]
[2026-02-18 10:00:04,362 - INFO - common - yaml file: params.yaml loaded successfully]
Found 116 images belonging to 2 classes.
8/8 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.8707 - loss: 2.1967
[2026-02-18 10:00:14,752 - INFO - common - json file saved at path: scores.json]
